# Llama-3.2-3B Necessary Circuit Analysis

A smaller restart notebook focused only on the Llama-3.2-3B necessary subcircuit. It loads the already-generated necessary-edge and activation-analysis artifacts and keeps the analysis narrow: edge roles, participating components, attention structure, readout deltas, and activation geometry.


In [19]:
from __future__ import annotations

import importlib
import json
import re
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
try:
    from IPython.display import HTML, Markdown, display
except Exception:
    class HTML(str):
        pass

    class Markdown(str):
        pass

    def display(*args, **kwargs):
        return None

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "scripts":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "animacy-circuit").is_dir():
    PROJECT_ROOT = PROJECT_ROOT / "animacy-circuit"

EXECUTABLE_DIR = PROJECT_ROOT / "scripts" / "executable"
if not (EXECUTABLE_DIR / "utils.py").exists():
    raise FileNotFoundError(f"Could not locate {EXECUTABLE_DIR / 'utils.py'}")
if str(EXECUTABLE_DIR) not in sys.path:
    sys.path.insert(0, str(EXECUTABLE_DIR))

import utils as inspection_utils
inspection_utils = importlib.reload(inspection_utils)

build_live_budgeted_circuit_figures = inspection_utils.build_live_budgeted_circuit_figures
budgeted_node_scores = inspection_utils.budgeted_node_scores

RESULTS_ROOT = PROJECT_ROOT / "results" / "eap_ig_localization"
MODEL_SLUG = "meta-llama_Llama-3.2-3B"
SEED = 42
RUN_NAME = None
ACTIVATION_RESULTS_DIR = None
CONCEPT_RESULTS_DIR = None
CONCEPT_MODE = "load"  # "load" uses saved concept-extraction artifacts; "compute" recomputes extraction only.
WHITE_BLUE_SCALE = [[0.0, "#ffffff"], [1.0, "#08519c"]]


def resolve_existing_path(value: str | Path | None) -> Path | None:
    if value is None:
        return None
    path = Path(value).expanduser()
    if path.is_absolute():
        return path
    candidates = [
        Path.cwd().resolve() / path,
        PROJECT_ROOT / path,
        PROJECT_ROOT.parent / path,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[-1]


def latest_report_dir() -> Path:
    preferred = sorted(RESULTS_ROOT.glob("necessary_edge_expansion_main_original_20_50_*"))
    fallback = sorted(RESULTS_ROOT.glob("necessary_edge_expansion_*"))
    candidates = preferred or fallback
    if not candidates:
        raise FileNotFoundError(f"No necessary-edge report found under {RESULTS_ROOT}")
    return candidates[-1]


def latest_activation_dir() -> Path:
    candidates = sorted(RESULTS_ROOT.glob("necessary_semantics_activations_*"))
    if not candidates:
        raise FileNotFoundError(f"No activation result bundle found under {RESULTS_ROOT}")
    return candidates[-1]


def latest_concept_dir(model_slug: str = MODEL_SLUG) -> Path:
    concept_root = PROJECT_ROOT / "results" / "concept_extraction" / model_slug
    if not concept_root.exists():
        raise FileNotFoundError(f"No concept-extraction root found under {concept_root}")
    candidates = [path for path in sorted(concept_root.iterdir()) if path.is_dir() and path.name != "smoke_test"]
    if not candidates:
        raise FileNotFoundError(f"No concept-extraction run directories found under {concept_root}")
    return candidates[-1]


def latest_matching_file(directory: Path, pattern: str) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda path: path.stat().st_mtime)
    return matches[-1] if matches else None


PLOT_OUTPUT_DIR: Path | None = None


def plot_output_dir() -> Path:
    if PLOT_OUTPUT_DIR is not None:
        output_dir = Path(PLOT_OUTPUT_DIR)
    elif "REPORT_DIR" in globals():
        output_dir = Path(REPORT_DIR) / "notebook_plots"
    else:
        output_dir = Path.cwd().resolve() / "notebook_plots"
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir


def plot_slug(fig, fallback: str = "plot") -> str:
    title = getattr(getattr(fig, "layout", None), "title", None)
    text = getattr(title, "text", None) or fallback
    slug = re.sub(r"[^A-Za-z0-9]+", "_", str(text)).strip("_").lower()
    return slug[:120] or fallback


def save_fig(fig, name: str | None = None) -> Path:
    output_dir = plot_output_dir()
    slug = plot_slug(fig, fallback=name or "plot") if name is None else plot_slug(fig, fallback=name)
    path = output_dir / f"{slug}.html"
    fig.write_html(path, full_html=True, include_plotlyjs="directory", config={"responsive": False})
    display(Markdown(f"Saved plot: `{path}`"))
    return path


def show_fig(fig) -> None:
    display(fig)


def parse_component(name: str) -> dict[str, Any]:
    if name == "input":
        return {"kind": "input", "layer": -1, "head": None}
    if name == "logits":
        return {"kind": "logits", "layer": -1, "head": None}
    mlp_match = re.fullmatch(r"m(\d+)", str(name))
    if mlp_match:
        return {"kind": "mlp", "layer": int(mlp_match.group(1)), "head": None}
    attn_match = re.fullmatch(r"a(\d+)\.h(\d+)", str(name))
    if attn_match:
        return {"kind": "attn", "layer": int(attn_match.group(1)), "head": int(attn_match.group(2))}
    return {"kind": "other", "layer": 0, "head": None}


def component_sort_key(name: str) -> tuple[int, int, int, str]:
    meta = parse_component(name)
    kind_order = {"attn": 0, "mlp": 1, "input": 2, "logits": 3, "other": 4}
    return (kind_order.get(meta["kind"], 5), int(meta["layer"]), int(meta["head"] or -1), str(name))


def edge_role(parent_type: str, child_type: str) -> str:
    if parent_type == "input" and child_type in {"attn", "mlp"}:
        return "input_to_component"
    if parent_type == "attn" and child_type == "mlp":
        return "head_to_mlp"
    if parent_type == "mlp" and child_type == "mlp":
        return "mlp_to_mlp"
    if child_type == "logits" and parent_type in {"attn", "mlp"}:
        return "component_to_logits"
    return "other"


REPORT_DIR = latest_report_dir()
ACTIVATION_DIR = resolve_existing_path(ACTIVATION_RESULTS_DIR) if ACTIVATION_RESULTS_DIR is not None else latest_activation_dir()
PLOT_OUTPUT_DIR = PROJECT_ROOT / "results" / "images" / "necessary_circuit_analysis" / REPORT_DIR.name
REPORT_DIR, ACTIVATION_DIR, PLOT_OUTPUT_DIR


(PosixPath('/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/eap_ig_localization/necessary_edge_expansion_main_original_20_50_2026-06-20'),
 PosixPath('/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/eap_ig_localization/necessary_semantics_activations_2026-06-21'),
 PosixPath('/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/images/necessary_circuit_analysis/necessary_edge_expansion_main_original_20_50_2026-06-20'))

## Load Llama-3.2-3B Artifacts


In [20]:
summary = pd.read_csv(REPORT_DIR / "necessary_budget_summary.csv")
collapsed = pd.read_csv(REPORT_DIR / "necessary_collapsed_edges.csv")
underlying = pd.read_csv(REPORT_DIR / "necessary_underlying_edges.csv")

attention_semantics = pd.read_csv(ACTIVATION_DIR / "attention_semantics.csv")
readout_semantics = pd.read_csv(ACTIVATION_DIR / "readout_semantics.csv")
activation_geometry_summary = pd.read_csv(ACTIVATION_DIR / "activation_geometry_summary.csv")
activation_pca_projection = pd.read_csv(ACTIVATION_DIR / "activation_pca_projection.csv")
target_metadata = pd.read_csv(ACTIVATION_DIR / "target_metadata.csv")

selected_summary = summary[summary["status"].isin(["selected", "selected_from_partial"])].copy()
selected_summary = selected_summary[selected_summary["model_slug"] == MODEL_SLUG].copy()
selected_summary = selected_summary[selected_summary["seed"] == SEED].copy()
if RUN_NAME is not None:
    selected_summary = selected_summary[selected_summary["run_name"] == RUN_NAME].copy()

if selected_summary.empty:
    raise ValueError(f"No selected Llama-3.2-3B slot found for model_slug={MODEL_SLUG!r}, seed={SEED}")

slot_row = selected_summary.sort_values(["run_name", "sample_size"]).iloc[0]
slot_cols = ["model_slug", "run_name", "sample_size", "seed", "selected_budget"]
slot_filter = {col: slot_row[col] for col in slot_cols}

def filter_slot(df: pd.DataFrame) -> pd.DataFrame:
    rows = df.copy()
    for key, value in slot_filter.items():
        if key in rows.columns:
            rows = rows[rows[key] == value]
    return rows.reset_index(drop=True)

llama_edges = filter_slot(collapsed)
llama_underlying = filter_slot(underlying)
llama_attention = filter_slot(attention_semantics)
llama_readout = filter_slot(readout_semantics)
llama_geometry = filter_slot(activation_geometry_summary)
llama_pca = filter_slot(activation_pca_projection)
llama_target_metadata = filter_slot(target_metadata)

print(f"Report dir: {REPORT_DIR}")
print(f"Activation dir: {ACTIVATION_DIR}")
print(f"Llama-3.2-3B slot: {slot_row['run_name']} / sample {slot_row['sample_size']} / seed {slot_row['seed']} / budget {slot_row['selected_budget']}")
print(f"Collapsed edges: {len(llama_edges)}")
print(f"Underlying edges: {len(llama_underlying)}")
print(f"Attention rows: {len(llama_attention)}")
print(f"Readout rows: {len(llama_readout)}")
print(f"Geometry rows: {len(llama_geometry)}")
print(f"PCA rows: {len(llama_pca)}")

display(selected_summary)
display(llama_target_metadata)


Report dir: /gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/eap_ig_localization/necessary_edge_expansion_main_original_20_50_2026-06-20
Activation dir: /gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/eap_ig_localization/necessary_semantics_activations_2026-06-21
Llama-3.2-3B slot: llama_seed_stability / sample 500 / seed 42 / budget 20
Collapsed edges: 20
Underlying edges: 24
Attention rows: 70
Readout rows: 192
Geometry rows: 96
PCA rows: 96000


,model_slug,run_name,sample_size,seed,threshold,status,edge_path,eval_path,partial_eval_path,eval_source,selected_budget,selected_faithfulness,selected_accuracy,selected_collapsed_edge_count,selected_underlying_edge_count
11,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,0.1,selected,animacy-circuit/results/eap_ig_localization/me...,animacy-circuit/results/eap_ig_localization/me...,animacy-circuit/results/eap_ig_localization/me...,complete,20,0.030803,0.04582,20,24


,model_slug,run_name,sample_size,seed,selected_budget,model_name,requested_model_name,target_source,target_filter_policy,target_settings_source,summary_path,example_count
0,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,meta-llama/Llama-3.2-3B,meta-llama/Llama-3.2-3B,NaN,model_success,repo_default,/gpfs/home4/spunzo/grammatical-circuits/animac...,500


## Selected Necessary Edges


In [21]:
underlying_edge_map = (
    llama_underlying.groupby("collapsed_edge", dropna=False)["underlying_edge"]
    .agg(lambda values: values.astype(str).tolist())
    .to_dict()
)

necessary_edges_table = llama_edges.copy()
necessary_edges_table["underlying_edges"] = necessary_edges_table["collapsed_edge"].map(underlying_edge_map)
necessary_edges_table = necessary_edges_table.sort_values("rank", kind="stable")[[
        "collapsed_edge",
        "parent",
        "child",
        "signed_sum",
        "abs_score",
        "rank",
        "underlying_edge_count",
        "underlying_edges",
    ]]

display(necessary_edges_table)

llama_circuit_nodes = budgeted_node_scores(llama_edges)
llama_circuit_figures = build_live_budgeted_circuit_figures(
    llama_edges,
    llama_circuit_nodes,
    top_k_edges=len(llama_edges),
    layered_circuit_min_height=500,
    layered_circuit_height_per_node=18,
)

display(Markdown("### Circuit View"))
display(Markdown("**Layered circuit**"))
display(llama_circuit_figures["layered_circuit"])

display(Markdown("**Residualized full circuit (path-length normalized routed load)**"))
display(llama_circuit_figures["residualized_normalized"])


,collapsed_edge,parent,child,signed_sum,abs_score,rank,underlying_edge_count,underlying_edges
0,input->m0,input,m0,-0.157078,0.157078,1,1,[input->m0]
1,m27->logits,m27,logits,-0.037448,0.037448,2,1,[m27->logits]
2,a27.h0->logits,a27.h0,logits,-0.027070,0.027070,3,1,[a27.h0->logits]
3,m0->m1,m0,m1,-0.022829,0.022829,4,1,[m0->m1]
4,m0->m2,m0,m2,-0.022203,0.022203,5,1,[m0->m2]
5,m2->m3,m2,m3,-0.019756,0.019756,6,1,[m2->m3]
6,m19->logits,m19,logits,-0.016702,0.016702,7,1,[m19->logits]
7,m26->logits,m26,logits,-0.015502,0.015502,8,1,[m26->logits]
8,m1->m2,m1,m2,-0.014823,0.014823,9,1,[m1->m2]
9,m22->logits,m22,logits,-0.013862,0.013862,10,1,[m22->logits]


### Circuit View

**Layered circuit**

**Residualized full circuit (path-length normalized routed load)**

## Edge Roles


In [22]:
edge_semantics = llama_edges.copy()
edge_semantics["parent_type"] = edge_semantics["parent"].map(lambda name: parse_component(str(name))["kind"])
edge_semantics["child_type"] = edge_semantics["child"].map(lambda name: parse_component(str(name))["kind"])
edge_semantics["parent_layer"] = edge_semantics["parent"].map(lambda name: parse_component(str(name))["layer"])
edge_semantics["child_layer"] = edge_semantics["child"].map(lambda name: parse_component(str(name))["layer"])
edge_semantics["edge_role"] = [
    edge_role(parent_type, child_type)
    for parent_type, child_type in zip(edge_semantics["parent_type"], edge_semantics["child_type"])
]
edge_semantics["layer_pair"] = edge_semantics["parent_layer"].astype(str) + "->" + edge_semantics["child_layer"].astype(str)

role_summary = (
    edge_semantics.groupby(["edge_role", "layer_pair"], dropna=False)
    .agg(
        edge_count=("collapsed_edge", "count"),
        total_abs_score=("abs_score", "sum"),
        signed_mass=("signed_sum", "sum"),
        top_examples=("collapsed_edge", lambda values: ", ".join(values.astype(str).head(4))),
    )
    .reset_index()
    .sort_values(["total_abs_score", "edge_count"], ascending=[False, False], kind="stable")
)

display(role_summary)

role_fig = px.bar(
    role_summary,
    x="edge_role",
    y="total_abs_score",
    color="layer_pair",
    hover_data=["edge_count", "signed_mass", "top_examples"],
    title="Llama-3.2-3B necessary circuit: edge-role mass",
)
role_fig.update_layout(xaxis_title="Edge role", yaxis_title="Total absolute score")
show_fig(role_fig)


,edge_role,layer_pair,edge_count,total_abs_score,signed_mass,top_examples
7,input_to_component,-1->0,1,0.157078,-0.157078,input->m0
6,component_to_logits,27->-1,3,0.077474,-0.077474,"m27->logits, a27.h0->logits, a27.h13->logits"
10,mlp_to_mlp,0->1,1,0.022829,-0.022829,m0->m1
11,mlp_to_mlp,0->2,1,0.022203,-0.022203,m0->m2
4,component_to_logits,25->-1,2,0.020543,-0.020543,"m25->logits, a25.h23->logits"
14,mlp_to_mlp,2->3,1,0.019756,-0.019756,m2->m3
0,component_to_logits,19->-1,1,0.016702,-0.016702,m19->logits
5,component_to_logits,26->-1,1,0.015502,-0.015502,m26->logits
13,mlp_to_mlp,1->2,1,0.014823,-0.014823,m1->m2
3,component_to_logits,22->-1,1,0.013862,-0.013862,m22->logits


## Participating Components


In [23]:
incident_rows = []
for edge in edge_semantics.to_dict("records"):
    for endpoint, direction in ((edge["parent"], "outgoing"), (edge["child"], "incoming")):
        meta = parse_component(str(endpoint))
        if meta["kind"] in {"input", "logits"}:
            continue
        incident_rows.append(
            {
                "component": str(endpoint),
                "component_type": meta["kind"],
                "layer": int(meta["layer"]),
                "head": meta["head"],
                "direction": direction,
                "edge_role": edge["edge_role"],
                "collapsed_edge": edge["collapsed_edge"],
                "abs_score": edge["abs_score"],
                "signed_sum": edge["signed_sum"],
                "rank": edge["rank"],
            }
        )

component_incidents = pd.DataFrame(incident_rows)
component_summary = (
    component_incidents.groupby(["component", "component_type", "layer", "head"], dropna=False)
    .agg(
        incident_edge_count=("collapsed_edge", "nunique"),
        abs_incident_mass=("abs_score", "sum"),
        signed_incident_mass=("signed_sum", "sum"),
        best_rank=("rank", "min"),
        top_roles=("edge_role", lambda values: ", ".join(pd.Series(values).value_counts().index[:3])),
    )
    .reset_index()
)

node_score_summary = llama_circuit_nodes.rename(columns={"node": "component"}).copy()
node_score_summary["component_type"] = node_score_summary["kind"].replace({"attention_head": "attn"})
node_score_summary["head"] = node_score_summary["head"].astype("Int64")
component_summary = component_summary.merge(
    node_score_summary[["component", "induced_score"]],
    on="component",
    how="left",
)
component_summary = component_summary.sort_values(["best_rank", "abs_incident_mass"], ascending=[True, False], kind="stable")
display(component_summary)

component_fig = px.bar(
    component_summary.sort_values(["component_type", "layer", "head"], kind="stable"),
    x="component",
    y="abs_incident_mass",
    color="component_type",
    hover_data=["incident_edge_count", "signed_incident_mass", "induced_score", "top_roles"],
    title="Llama-3.2-3B necessary circuit: component incident mass",
)
component_fig.update_layout(xaxis_title="Component", yaxis_title="Absolute incident mass")
show_fig(component_fig)

attn_heatmap = (
    node_score_summary.loc[node_score_summary["component_type"] == "attn", ["layer", "head", "induced_score"]]
    .assign(component_label=lambda frame: "H" + frame["head"].astype(int).astype(str))
    .pivot(index="component_label", columns="layer", values="induced_score")
)
mlp_heatmap = (
    node_score_summary.loc[node_score_summary["component_type"] == "mlp", ["layer", "induced_score"]]
    .assign(component_label="MLP")
    .pivot(index="component_label", columns="layer", values="induced_score")
)
heatmap_layers = sorted(
    set(attn_heatmap.columns.tolist()) | set(mlp_heatmap.columns.tolist())
)
head_labels = sorted(
    attn_heatmap.index.tolist(),
    key=lambda label: int(str(label).removeprefix("H")),
)
component_heatmap = pd.concat([attn_heatmap, mlp_heatmap], axis=0)
component_heatmap = component_heatmap.reindex(index=[*head_labels, "MLP"], columns=heatmap_layers)

heatmap_raw_values = component_heatmap.to_numpy(dtype=float).ravel() if not component_heatmap.empty else np.array([], dtype=float)
positive_heatmap_values = heatmap_raw_values[np.isfinite(heatmap_raw_values) & (heatmap_raw_values > 0)]
log_score_offset = float(np.nanmin(positive_heatmap_values)) if positive_heatmap_values.size else 1.0


def log_induced_score_matrix(frame: pd.DataFrame) -> np.ndarray:
    values = frame.to_numpy(dtype=float)
    return np.where(
        np.isfinite(values) & (values > 0),
        np.log10(1.0 + values / log_score_offset),
        0.0,
    )


component_heatmap_log = log_induced_score_matrix(component_heatmap)
log_zmax = float(np.nanmax(component_heatmap_log)) if component_heatmap_log.size else 0.0
log_zmax = log_zmax if log_zmax > 0 else 1.0

heatmap_fig = go.Figure(
    data=go.Heatmap(
        z=component_heatmap_log,
        x=[f"L{int(layer)}" for layer in component_heatmap.columns],
        y=component_heatmap.index.tolist(),
        customdata=component_heatmap.to_numpy(),
        colorscale=WHITE_BLUE_SCALE,
        zmin=0.0,
        zmax=log_zmax,
        colorbar=dict(title="log induced score"),
        hovertemplate="Layer %{x}<br>Component %{y}<br>Induced score %{customdata:.4f}<br>Log color %{z:.4f}<extra></extra>",
    )
)
heatmap_fig.update_xaxes(title_text="Layer")
heatmap_fig.update_yaxes(title_text="Component")
heatmap_fig.update_layout(
    title="Llama-3.2-3B necessary circuit: induced score heatmap for heads and MLPs",
    height=max(520, 26 * max(len(component_heatmap.index), 1) + 180),
)
show_fig(heatmap_fig)


,component,component_type,layer,head,incident_edge_count,abs_incident_mass,signed_incident_mass,best_rank,top_roles,induced_score
5,m0,mlp,0,NaN,5,0.223555,-0.222786,1,"mlp_to_mlp, input_to_component, other",0.223555
14,m27,mlp,27,NaN,1,0.037448,-0.037448,2,component_to_logits,0.037448
2,a27.h0,attn,27,0.0,1,0.027070,-0.027070,3,component_to_logits,0.027070
6,m1,mlp,1,NaN,3,0.050470,-0.050470,4,"mlp_to_mlp, input_to_component",0.050470
8,m2,mlp,2,NaN,5,0.079972,-0.077094,5,"mlp_to_mlp, other, input_to_component",0.079972
15,m3,mlp,3,NaN,2,0.028158,-0.028158,6,mlp_to_mlp,0.028158
7,m19,mlp,19,NaN,1,0.016702,-0.016702,7,component_to_logits,0.016702
13,m26,mlp,26,NaN,1,0.015502,-0.015502,8,component_to_logits,0.015502
11,m22,mlp,22,NaN,1,0.013862,-0.013862,10,component_to_logits,0.013862
0,a1.h3,attn,1,3.0,1,0.013042,-0.012274,11,other,0.013042


## Attention Pattern Analysis For Necessary Heads

Load precomputed attention patterns for necessary heads. This notebook does not run the model for this section; generate or refresh the CSV on a GPU with `sbatch jobs/job.sh`. Rows are query positions and columns are key positions in the sentence template `The [patient] was [verb p.p] by the `.


In [18]:
SENTENCE_POSITION_ORDER = ["BOS", "The", "patient", "was", "verb", "by", "the"]
MODEL_DISPLAY_NAME = {
    "gpt2": "GPT-2",
    "meta-llama_Llama-3.2-3B": "Llama-3.2-3B",
    "Qwen_Qwen3-4B": "Qwen3-4B",
    "google_gemma-3-4b-pt": "Gemma-3-4B-PT",
}.get(MODEL_SLUG, MODEL_SLUG)


def latest_attention_pattern_file(model_slug: str = MODEL_SLUG):
    candidates = sorted(
        [*RESULTS_ROOT.glob("necessary_attention_patterns_*"), *RESULTS_ROOT.glob("necessary_circuit_diagnostics_*")],
        key=lambda path: path.stat().st_mtime,
    )
    for directory in reversed(candidates):
        attention_path = directory / "attention_pattern_summary.csv"
        if not attention_path.exists():
            continue
        try:
            slugs = set(pd.read_csv(attention_path, usecols=["model_slug"])["model_slug"].dropna().astype(str).unique())
        except Exception:
            continue
        if model_slug in slugs:
            return attention_path
    return None


attention_pattern_path = latest_attention_pattern_file()
if attention_pattern_path is None:
    attention_pattern_summary = pd.DataFrame()
    display(Markdown(
        "No precomputed attention-pattern CSV found for this model. Run "
        "`sbatch jobs/job.sh` from the repository root, then rerun this cell."
    ))
else:
    attention_pattern_summary = filter_slot(pd.read_csv(attention_pattern_path))
    necessary_heads = sorted(
        component_summary.loc[component_summary["component_type"] == "attn", "component"].dropna().astype(str).unique().tolist(),
        key=component_sort_key,
    )
    available_heads = [
        head for head in necessary_heads
        if head in set(attention_pattern_summary["component"].dropna().astype(str))
    ]
    if attention_pattern_summary.empty or not available_heads:
        display(Markdown(
            f"`{attention_pattern_path}` exists, but it has no rows for the selected slot "
            f"`{MODEL_SLUG}` / seed `{SEED}` / budget `{slot_row['selected_budget']}`. "
            "Regenerate it with `sbatch jobs/job.sh`."
        ))
    else:
        display(pd.DataFrame([{
            "attention_pattern_path": str(attention_pattern_path),
            "rows": len(attention_pattern_summary),
            "heads": len(available_heads),
        }]))

        heads_per_condition_row = 3
        condition_blocks = [
            ("clean", "Clean", "coloraxis"),
            ("corrupt", "Corrupt", "coloraxis2"),
        ]
        head_rows = int(np.ceil(len(available_heads) / heads_per_condition_row))
        spacer_rows = 1
        total_rows = head_rows * len(condition_blocks) + spacer_rows
        total_cols = heads_per_condition_row
        spacer_row = head_rows + 1
        row_heights = [1.0] * head_rows + [0.28] + [1.0] * head_rows

        subplot_titles = []
        for block_index, (_condition, _label, _coloraxis) in enumerate(condition_blocks):
            if block_index == 1:
                subplot_titles.extend([""] * heads_per_condition_row)
            for row_index in range(head_rows):
                row_heads = available_heads[
                    row_index * heads_per_condition_row : (row_index + 1) * heads_per_condition_row
                ]
                subplot_titles.extend([*row_heads, *([""] * (heads_per_condition_row - len(row_heads)))])

        pattern_fig = make_subplots(
            rows=total_rows,
            cols=total_cols,
            subplot_titles=subplot_titles,
            row_heights=row_heights,
            horizontal_spacing=0.055,
            vertical_spacing=0.05,
        )

        for head_index, component in enumerate(available_heads):
            head_row_index = head_index // heads_per_condition_row + 1
            col_index = head_index % heads_per_condition_row + 1
            for block_index, (condition, condition_label, coloraxis_name) in enumerate(condition_blocks):
                row_index = head_row_index if block_index == 0 else spacer_row + head_row_index
                matrix = (
                    attention_pattern_summary[
                        (attention_pattern_summary["component"].astype(str) == component)
                        & (attention_pattern_summary["condition"].astype(str) == condition)
                    ]
                    .pivot(index="query_token", columns="key_token", values="attention_mass_mean")
                    .reindex(index=SENTENCE_POSITION_ORDER, columns=SENTENCE_POSITION_ORDER)
                )
                pattern_fig.add_trace(
                    go.Heatmap(
                        z=matrix.to_numpy(),
                        x=matrix.columns.tolist(),
                        y=matrix.index.tolist(),
                        coloraxis=coloraxis_name,
                        hovertemplate=(
                            "condition=" + condition_label + "<br>"
                            "head=" + component + "<br>"
                            "query=%{y}<br>key=%{x}<br>attention=%{z:.4f}<extra></extra>"
                        ),
                    ),
                    row=row_index,
                    col=col_index,
                )
                pattern_fig.update_xaxes(
                    categoryorder="array",
                    categoryarray=SENTENCE_POSITION_ORDER,
                    showline=True,
                    mirror=True,
                    linewidth=1.5,
                    linecolor="#222222",
                    ticks="outside",
                    row=row_index,
                    col=col_index,
                )
                pattern_fig.update_yaxes(
                    categoryorder="array",
                    categoryarray=SENTENCE_POSITION_ORDER,
                    autorange="reversed",
                    showline=True,
                    mirror=True,
                    linewidth=1.5,
                    linecolor="#222222",
                    ticks="outside",
                    row=row_index,
                    col=col_index,
                )

        pattern_height = max(260 * (head_rows * len(condition_blocks)) + 300, 1180)
        pattern_fig.update_layout(
            title=dict(
                text=f"{MODEL_DISPLAY_NAME} necessary heads: precomputed mean attention patterns over sentence-template tokens",
                x=0.5,
                xanchor="center",
                font=dict(size=24),
            ),
            coloraxis=dict(
                colorscale=WHITE_BLUE_SCALE,
                cmin=0.0,
                cmax=1.0,
                colorbar=dict(title="mean attention", x=1.02, y=0.80, len=0.32),
            ),
            coloraxis2=dict(
                colorscale=WHITE_BLUE_SCALE,
                cmin=0.0,
                cmax=1.0,
                colorbar=dict(title="mean attention", x=1.02, y=0.23, len=0.32),
            ),
            height=pattern_height,
            margin=dict(l=55, r=110, t=200, b=45),
        )
        pattern_fig.add_annotation(
            text="Clean",
            x=0.5,
            y=1.05,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(size=21),
        )
        pattern_fig.add_annotation(
            text="Corrupt",
            x=0.5,
            y=0.49,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(size=21),
        )
        display(pattern_fig)
        display(attention_pattern_summary)

# Preserve the model-specific name used elsewhere during ad hoc notebook work.
globals()[f"{MODEL_SLUG.split('/')[-1].replace('-', '_').replace('.', '_')}_attention_pattern_summary"] = attention_pattern_summary


,attention_pattern_path,rows,heads
0,/gpfs/home4/spunzo/grammatical-circuits/animac...,490,5


,model_slug,run_name,sample_size,seed,selected_budget,component,component_type,layer,head,condition,query_token,key_token,attention_mass_mean,example_count
0,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,a1.h3,attn,1,3,clean,BOS,BOS,1.000000,500
1,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,a1.h3,attn,1,3,clean,BOS,The,0.000000,500
2,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,a1.h3,attn,1,3,clean,BOS,patient,0.000000,500
3,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,a1.h3,attn,1,3,clean,BOS,was,0.000000,500
4,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,a1.h3,attn,1,3,clean,BOS,verb,0.000000,500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
485,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,a27.h13,attn,27,13,corrupt,the,patient,0.000847,500
486,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,a27.h13,attn,27,13,corrupt,the,was,0.007231,500
487,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,a27.h13,attn,27,13,corrupt,the,verb,0.023790,500
488,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,a27.h13,attn,27,13,corrupt,the,by,0.743597,500


## Residual Stream Verb Activation PCA

Load the residual-stream PCA artifact and show clean/corrupt verb-token activations for every layer and residual stream site. Rows are layers; columns are `hook_resid_pre`, `hook_resid_mid`, and `hook_resid_post`.


In [24]:
MODEL_DISPLAY_NAME = {
    "gpt2": "GPT-2",
    "meta-llama_Llama-3.2-3B": "Llama-3.2-3B",
    "Qwen_Qwen3-4B": "Qwen3-4B",
    "google_gemma-3-4b-pt": "Gemma-3-4B-PT",
}.get(MODEL_SLUG, MODEL_SLUG)
RESIDUAL_HOOK_ORDER = ["pre", "mid", "post"]
RESIDUAL_HOOK_LABELS = {"pre": "resid pre", "mid": "resid mid", "post": "resid post"}
RESIDUAL_CLASS_COLORS = {
    "animate-compatible verb": "#1f77b4",
    "inanimate-compatible verb": "#d62728",
}


STATIC_RESIDUAL_DPI = 150
STATIC_RESIDUAL_COL_WIDTH = 4.0
STATIC_RESIDUAL_ROW_HEIGHT = 2.35
STATIC_RESIDUAL_PREVIEW_MAX_WIDTH = 1400
STATIC_RESIDUAL_PREVIEW_MAX_HEIGHT = 5000


def resolve_concept_analysis_dir():
    if CONCEPT_RESULTS_DIR is not None:
        path = resolve_existing_path(CONCEPT_RESULTS_DIR)
        return path if path is not None and path.exists() else None
    try:
        return latest_concept_dir(MODEL_SLUG)
    except FileNotFoundError as exc:
        display(Markdown(f"No concept-analysis artifact directory found for `{MODEL_SLUG}`: {exc}"))
        return None


def save_residual_pca_static(residual_pca_points: pd.DataFrame, layers: list[int], hook_points: list[str]) -> Path:
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    title = f"{MODEL_DISPLAY_NAME}: residual stream PCA of verb-token activations"
    output_dir = plot_output_dir() if "plot_output_dir" in globals() else Path.cwd().resolve() / "notebook_plots"
    output_dir.mkdir(parents=True, exist_ok=True)
    slug = re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_").lower()[:120]
    path = output_dir / f"{slug}.png"

    fig_width = max(13.0, STATIC_RESIDUAL_COL_WIDTH * len(hook_points) + 1.1)
    fig_height = max(7.0, STATIC_RESIDUAL_ROW_HEIGHT * len(layers) + 1.4)
    fig, axes = plt.subplots(
        len(layers),
        len(hook_points),
        figsize=(fig_width, fig_height),
        squeeze=False,
        sharex=False,
        sharey=False,
    )

    for row_index, layer in enumerate(layers):
        for col_index, hook_point in enumerate(hook_points):
            ax = axes[row_index, col_index]
            panel = residual_pca_points[
                (residual_pca_points["layer"] == layer)
                & (residual_pca_points["hook_point"] == hook_point)
            ]
            for verb_class, class_rows in panel.groupby("verb_class", sort=False):
                ax.scatter(
                    class_rows["pc1"],
                    class_rows["pc2"],
                    s=7,
                    alpha=0.58,
                    linewidths=0,
                    color=RESIDUAL_CLASS_COLORS.get(str(verb_class), "#666666"),
                )
            ax.set_title(f"L{layer} {RESIDUAL_HOOK_LABELS.get(hook_point, hook_point)}", fontsize=9, pad=8)
            ax.grid(True, color="#dddddd", linewidth=0.5, alpha=0.7)
            ax.tick_params(axis="both", labelsize=7)
            if row_index == len(layers) - 1:
                ax.set_xlabel("PC1", fontsize=9)
            if col_index == 0:
                ax.set_ylabel("PC2", fontsize=9)

    handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="",
            label=label,
            markerfacecolor=color,
            markeredgecolor="none",
            markersize=7,
            alpha=0.75,
        )
        for label, color in RESIDUAL_CLASS_COLORS.items()
    ]
    fig.suptitle(title, x=0.02, y=0.995, ha="left", fontsize=14)
    fig.legend(handles=handles, title="Verb class", loc="lower center", ncol=len(handles), frameon=False, fontsize=10, title_fontsize=10)
    fig.tight_layout(rect=(0.0, 0.025, 1.0, 0.985), h_pad=1.4, w_pad=1.0)
    fig.savefig(path, dpi=STATIC_RESIDUAL_DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return path


def save_residual_pca_preview(path: Path) -> Path:
    from PIL import Image

    preview_path = path.with_name(f"{path.stem}_preview{path.suffix}")
    with Image.open(path) as image:
        scale = min(
            1.0,
            STATIC_RESIDUAL_PREVIEW_MAX_WIDTH / image.width,
            STATIC_RESIDUAL_PREVIEW_MAX_HEIGHT / image.height,
        )
        if scale < 1.0:
            preview_size = (max(1, int(image.width * scale)), max(1, int(image.height * scale)))
            image = image.resize(preview_size, Image.Resampling.LANCZOS)
        image.save(preview_path, optimize=True)
    return preview_path


concept_analysis_dir = resolve_concept_analysis_dir()
residual_pca_path = (
    latest_matching_file(concept_analysis_dir, "residual_stream_pca_points_*.csv")
    if concept_analysis_dir is not None
    else None
)
residual_pca_summary_path = (
    latest_matching_file(concept_analysis_dir, "residual_stream_pca_summary_*.json")
    if concept_analysis_dir is not None
    else None
)

if residual_pca_path is None:
    residual_pca_points = pd.DataFrame()
    display(Markdown(
        "No saved residual-stream PCA artifact found. Generate it on a GPU with "
        "`sbatch --export=ALL,MODEL_SLUG="
        f"{MODEL_SLUG} jobs/residual_stream_pca.sh` and rerun this cell."
    ))
else:
    residual_pca_points = pd.read_csv(residual_pca_path).copy()
    residual_pca_summary = (
        json.loads(residual_pca_summary_path.read_text())
        if residual_pca_summary_path is not None
        else {}
    )
    residual_pca_points["verb_class"] = residual_pca_points["sentence_type"].map({
        "clean": "animate-compatible verb",
        "corrupt": "inanimate-compatible verb",
    }).fillna(residual_pca_points["sentence_type"].astype(str))
    residual_pca_points["hook_point"] = residual_pca_points["hook_point"].astype(str)
    residual_pca_points["layer"] = residual_pca_points["layer"].astype(int)

    layers = sorted(residual_pca_points["layer"].dropna().astype(int).unique().tolist())
    hook_points = [hook for hook in RESIDUAL_HOOK_ORDER if hook in set(residual_pca_points["hook_point"])]
    if not layers or not hook_points:
        display(Markdown(f"`{residual_pca_path}` did not contain residual-stream PCA rows to plot."))
    else:
        display(pd.DataFrame([{
            "pca_points_path": str(residual_pca_path),
            "summary_path": str(residual_pca_summary_path) if residual_pca_summary_path else None,
            "test_examples": residual_pca_summary.get("test_examples"),
            "hook_count": residual_pca_summary.get("hook_count"),
            "rows": len(residual_pca_points),
        }]))
        residual_pca_png_path = save_residual_pca_static(residual_pca_points, layers, hook_points)
        residual_pca_preview_path = save_residual_pca_preview(residual_pca_png_path)
        display(Markdown(
            f"Saved static residual-stream PCA plot: `{residual_pca_png_path}`  
"
            f"Displaying notebook preview: `{residual_pca_preview_path}`"
        ))
        try:
            from IPython.display import Image as IPythonImage
            display(IPythonImage(filename=str(residual_pca_preview_path)))
        except Exception as exc:
            display(Markdown(f"Could not display `{residual_pca_preview_path}` inline: {exc}"))

,pca_points_path,summary_path,test_examples,hook_count,rows
0,/gpfs/home4/spunzo/grammatical-circuits/animac...,/gpfs/home4/spunzo/grammatical-circuits/animac...,947,84,159096


Saved plot: `/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/images/necessary_circuit_analysis/necessary_edge_expansion_main_original_20_50_2026-06-20/llama_3_2_3b_residual_stream_pca_of_verb_token_activations.html`

## Animacy Concept Extraction

This section separates the raw extraction magnitude `concept_norm` from the saved validation-based site selection in the standalone concept notebook. By default it loads the separately computed concept-extraction artifacts under `results/concept_extraction/meta-llama_Llama-3.2-3B/...`; set `CONCEPT_MODE = "compute"` in the setup cell to recompute extraction on the fly from the saved train split without running any steering.


In [8]:
import gc
import torch

from circuit_finder_core import extract_animacy_concept_vectors, load_model_context

concept_dir = resolve_existing_path(CONCEPT_RESULTS_DIR) if CONCEPT_RESULTS_DIR is not None else latest_concept_dir()
concept_summary_path = latest_matching_file(concept_dir, "concept_extraction_summary_*.json")
concept_site_summary_path = latest_matching_file(concept_dir, "concept_site_summary_*.csv")
concept_vector_path = latest_matching_file(concept_dir, "concept_vectors_*.pt")
concept_train_path = latest_matching_file(concept_dir, "train_split_*.csv")
concept_selected_path = latest_matching_file(concept_dir, "selected_site_*.json")
concept_validation_path = latest_matching_file(concept_dir, "validation_sweep_*.csv")

concept_summary_payload = json.loads(concept_summary_path.read_text()) if concept_summary_path is not None else {}
concept_config = concept_summary_payload.get("config", {}) or {}
concept_hook_points = list(concept_config.get("hook_points", ["hook_resid_pre", "hook_resid_mid", "hook_resid_post"]))
concept_extraction_batch_size = int(concept_config.get("extraction_batch_size", 64))
concept_model_name = str(concept_summary_payload.get("model_name", MODEL_SLUG))

if CONCEPT_MODE == "load":
    if concept_site_summary_path is None or concept_vector_path is None:
        raise FileNotFoundError(f"Missing saved concept-extraction artifacts under {concept_dir}")
    concept_site_summary = pd.read_csv(concept_site_summary_path)
    concept_vector_payload = torch.load(concept_vector_path, map_location="cpu")
    concept_vectors = {hook_name: vector.float().cpu() for hook_name, vector in concept_vector_payload["vectors"].items()}
    concept_source_text = f"loaded saved extraction from {concept_dir}"
elif CONCEPT_MODE == "compute":
    if concept_train_path is None:
        raise FileNotFoundError(f"Missing train split under {concept_dir}; cannot recompute extraction.")
    concept_train_split = pd.read_csv(concept_train_path)
    concept_context = load_model_context(PROJECT_ROOT, concept_model_name)
    concept_model = concept_context["model"]
    concept_vectors, concept_site_summary = extract_animacy_concept_vectors(
        model=concept_model,
        train_df=concept_train_split,
        hook_points=concept_hook_points,
        batch_size=concept_extraction_batch_size,
    )
    concept_vectors = {hook_name: vector.float().cpu() for hook_name, vector in concept_vectors.items()}
    concept_source_text = (
        f"computed on the fly from {concept_train_path.name} using hook points {', '.join(concept_hook_points)}"
    )
    del concept_model, concept_context
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    raise ValueError(f"Unsupported CONCEPT_MODE={CONCEPT_MODE!r}; expected 'load' or 'compute'.")

concept_selected_payload = json.loads(concept_selected_path.read_text()) if concept_selected_path is not None else {}
concept_validation = pd.read_csv(concept_validation_path) if concept_validation_path is not None else pd.DataFrame()

concept_site_summary = concept_site_summary.copy()
concept_site_summary["hook_point_label"] = concept_site_summary["hook_point"].astype(str).str.replace("hook_", "", regex=False)
concept_site_summary = concept_site_summary.sort_values(
    ["concept_norm", "layer", "hook_point"],
    ascending=[False, True, True],
    kind="stable",
).reset_index(drop=True)

hook_point_order = ["resid_pre", "resid_mid", "resid_post"]
concept_heatmap = (
    concept_site_summary.pivot(index="hook_point_label", columns="layer", values="concept_norm")
    .reindex(index=hook_point_order)
)
concept_heatmap = concept_heatmap.reindex(sorted(concept_heatmap.columns.tolist()), axis=1)
concept_zmax = float(np.nanmax(concept_heatmap.to_numpy())) if np.isfinite(concept_heatmap.to_numpy()).any() else 1.0

top_concept_site = concept_site_summary.iloc[0].to_dict()
top_concept_hook = str(top_concept_site["hook_name"])
top_concept_vector = concept_vectors[top_concept_hook]
selected_site = (concept_summary_payload.get("selected") or concept_selected_payload.get("selected") or {})
selected_hook = str(selected_site.get("hook_name")) if selected_site.get("hook_name") is not None else None
selected_vector = concept_vectors.get(selected_hook) if selected_hook is not None else None

display(Markdown(f"**Concept source:** `{concept_source_text}`"))
display(pd.DataFrame([{
    "mode": CONCEPT_MODE,
    "concept_dir": str(concept_dir),
    "train_examples": int(top_concept_site["train_examples"]),
    "hook_points": ", ".join(concept_hook_points),
    "top_concept_norm_hook": top_concept_hook,
    "top_concept_norm": float(top_concept_site["concept_norm"]),
    "selected_hook": selected_hook,
    "selected_alpha": selected_site.get("alpha"),
    "selected_signed_effect_mean": selected_site.get("signed_effect_mean"),
    "selected_concept_norm": selected_site.get("concept_norm"),
    "selected_vector_dim": int(selected_vector.shape[0]) if selected_vector is not None else None,
}]))

if selected_site:
    display(Markdown("**Saved validation-based selected site**"))
    display(pd.DataFrame([selected_site]))

display(concept_site_summary[[
    "layer", "hook_point", "hook_name", "site_key", "train_examples", "clean_mean_norm", "corrupt_mean_norm", "concept_norm"
]])

concept_fig = go.Figure(
    go.Heatmap(
        z=concept_heatmap.to_numpy(),
        x=concept_heatmap.columns.tolist(),
        y=concept_heatmap.index.tolist(),
        colorscale=WHITE_BLUE_SCALE,
        zmin=0.0,
        zmax=concept_zmax,
        hovertemplate="hook=%{y}<br>layer=%{x}<br>concept_norm=%{z:.4f}<extra></extra>",
    )
)
concept_fig.update_layout(
    title="Llama-3.2-3B animacy concept extraction: concept norm by layer and hook site",
    xaxis_title="Layer",
    yaxis_title="Hook site",
    height=340,
    margin=dict(l=40, r=40, t=70, b=40),
)
show_fig(concept_fig)

if not concept_validation.empty:
    validation_best_idx = concept_validation.groupby(["layer", "hook_point"])["signed_effect_mean"].idxmax()
    validation_best = concept_validation.loc[validation_best_idx].copy()
    validation_best["hook_point_label"] = validation_best["hook_point"].astype(str).str.replace("hook_", "", regex=False)
    validation_heatmap = (
        validation_best.pivot(index="hook_point_label", columns="layer", values="signed_effect_mean")
        .reindex(index=hook_point_order)
    )
    validation_heatmap = validation_heatmap.reindex(sorted(validation_heatmap.columns.tolist()), axis=1)
    display(Markdown("**Best validation steering effect by layer and hook site from the saved concept run**"))
    validation_fig = go.Figure(
        go.Heatmap(
            z=validation_heatmap.to_numpy(),
            x=validation_heatmap.columns.tolist(),
            y=validation_heatmap.index.tolist(),
            colorscale="RdBu_r",
            zmid=0.0,
            hovertemplate="hook=%{y}<br>layer=%{x}<br>best signed effect=%{z:.4f}<extra></extra>",
        )
    )
    validation_fig.update_layout(
        title="Llama-3.2-3B saved validation selection surface",
        xaxis_title="Layer",
        yaxis_title="Hook site",
        height=340,
        margin=dict(l=40, r=40, t=70, b=40),
    )
    show_fig(validation_fig)
    display(validation_best[["layer", "hook_point", "hook_name", "alpha", "concept_norm", "signed_effect_mean", "selection_eligible"]].sort_values(["signed_effect_mean", "layer"], ascending=[False, True], kind="stable"))
else:
    display(Markdown("Validation-effect ranking is unavailable in `compute` mode because this notebook is only recomputing extraction, not the steering sweep."))


**Concept source:** `loaded saved extraction from /gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/concept_extraction/meta-llama_Llama-3.2-3B/2026-06-27`

,mode,concept_dir,train_examples,hook_points,top_concept_norm_hook,top_concept_norm,selected_hook,selected_alpha,selected_signed_effect_mean,selected_concept_norm,selected_vector_dim
0,load,/gpfs/home4/spunzo/grammatical-circuits/animac...,2840,"hook_resid_pre, hook_resid_mid, hook_resid_post",blocks.27.hook_resid_post,16.952997,None,None,None,None,None


,layer,hook_point,hook_name,site_key,train_examples,clean_mean_norm,corrupt_mean_norm,concept_norm
0,27,hook_resid_post,blocks.27.hook_resid_post,blocks__27__hook_resid_post,2840,52.972218,53.767075,16.952997
1,26,hook_resid_post,blocks.26.hook_resid_post,blocks__26__hook_resid_post,2840,36.513195,37.662605,14.750183
2,27,hook_resid_pre,blocks.27.hook_resid_pre,blocks__27__hook_resid_pre,2840,36.513195,37.662605,14.750183
3,27,hook_resid_mid,blocks.27.hook_resid_mid,blocks__27__hook_resid_mid,2840,41.955124,43.035530,14.746161
4,26,hook_resid_mid,blocks.26.hook_resid_mid,blocks__26__hook_resid_mid,2840,37.817673,38.695477,13.757853
...,...,...,...,...,...,...,...,...
79,1,hook_resid_mid,blocks.1.hook_resid_mid,blocks__1__hook_resid_mid,2840,1.755515,1.905011,0.944984
80,0,hook_resid_post,blocks.0.hook_resid_post,blocks__0__hook_resid_post,2840,1.338720,1.488971,0.921939
81,1,hook_resid_pre,blocks.1.hook_resid_pre,blocks__1__hook_resid_pre,2840,1.338720,1.488971,0.921939
82,0,hook_resid_mid,blocks.0.hook_resid_mid,blocks__0__hook_resid_mid,2840,0.599956,0.623865,0.466426


Saved plot: `/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/eap_ig_localization/necessary_edge_expansion_main_original_20_50_2026-06-20/notebook_plots/llama_3_2_3b_animacy_concept_extraction_concept_norm_by_layer_and_hook_site.html`

Validation-effect ranking is unavailable in `compute` mode because this notebook is only recomputing extraction, not the steering sweep.